In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../dataset/df_working_cleaned.parquet')

In [2]:
df = df[df['allocated_processors'] > 0]

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6452617 entries, 0 to 6452616
Data columns (total 11 columns):
 #   Column                Dtype
---  ------                -----
 0   requested_processors  int64
 1   requested_time        int64
 2   requested_memory      int64
 3   queue_number          int64
 4   preceding_job_number  int64
 5   think_time            int64
 6   wait_time             int64
 7   submit_time           int64
 8   run_time              int64
 9   allocated_processors  int64
 10  user_id               int64
dtypes: int64(11)
memory usage: 541.5 MB


In [4]:
df.describe()

,requested_processors,requested_time,requested_memory,queue_number,preceding_job_number,think_time,wait_time,submit_time,run_time,allocated_processors,user_id
count,6.452617e+06,6.452617e+06,6452617.0,6.452617e+06,6452617.0,6452617.0,6.452617e+06,6.452617e+06,6.452617e+06,6.452617e+06,6.452617e+06
mean,4.295811e+00,8.155702e+04,-1.0,2.745776e+00,-1.0,-1.0,1.253623e+04,1.434609e+08,5.909243e+03,4.295811e+00,1.034230e+02
std,1.504043e+01,1.217488e+06,0.0,1.312119e+00,0.0,0.0,7.169695e+04,8.006313e+07,2.380786e+04,1.504043e+01,6.938906e+01
min,1.000000e+00,0.000000e+00,-1.0,1.000000e+00,-1.0,-1.0,0.000000e+00,2.146000e+03,0.000000e+00,1.000000e+00,1.000000e+00
25%,1.000000e+00,7.140000e+03,-1.0,2.000000e+00,-1.0,-1.0,1.100000e+01,8.819136e+07,8.000000e+00,1.000000e+00,6.000000e+01
50%,1.000000e+00,9.000000e+03,-1.0,2.000000e+00,-1.0,-1.0,4.600000e+01,1.028810e+08,1.250000e+02,1.000000e+00,9.400000e+01
75%,1.000000e+00,1.620000e+04,-1.0,3.000000e+00,-1.0,-1.0,2.787000e+03,2.234237e+08,1.879000e+03,1.000000e+00,1.350000e+02
max,5.120000e+02,3.600000e+07,-1.0,6.000000e+00,-1.0,-1.0,1.057163e+07,2.878164e+08,8.989324e+06,5.120000e+02,2.570000e+02


### Feature Engineering - 'cpu_utilization'

In [5]:
df['actual_start_time'] = df['submit_time'] + df['wait_time']
df['actual_end_time'] = df['actual_start_time'] + df['run_time']

In [6]:
df = df.sort_values(by=['user_id', 'submit_time'])
df['prev_job_end'] = df.groupby('user_id')['actual_end_time'].shift(1)
df['think_time'] = df['submit_time'] - df['prev_job_end']
df['think_time'] = df['think_time'].fillna(0).clip(lower=0)

In [7]:
starts_cpu = pd.DataFrame({'time': df['actual_start_time'], 'cpu_change': df['allocated_processors']})
ends_cpu = pd.DataFrame({'time': df['actual_end_time'], 'cpu_change': -df['allocated_processors']})

In [8]:
submits_queue = pd.DataFrame({'time': df['submit_time'], 'queue_change': 1})
starts_queue = pd.DataFrame({'time': df['actual_start_time'], 'queue_change': -1})

In [9]:
cpu_events = pd.concat([starts_cpu, ends_cpu]).sort_values('time')
cpu_events = cpu_events.groupby('time')['cpu_change'].sum().reset_index()
cpu_events['current_cpu_utilization'] = cpu_events['cpu_change'].cumsum()

In [10]:
queue_events = pd.concat([submits_queue, starts_queue]).sort_values('time')
queue_events = queue_events.groupby('time')['queue_change'].sum().reset_index()
queue_events['preceding_job_number'] = queue_events['queue_change'].cumsum()

In [11]:
df = df.sort_values('submit_time')

df = pd.merge_asof(df, cpu_events[['time', 'current_cpu_utilization']], 
                   left_on='submit_time', right_on='time', direction='backward')

if 'preceding_job_number' in df.columns:
    df = df.drop(columns=['preceding_job_number'])

df = pd.merge_asof(df, queue_events[['time', 'preceding_job_number']], 
                   left_on='submit_time', right_on='time', direction='backward')

In [12]:
df['current_cpu_utilization'] = df['current_cpu_utilization'].fillna(0)
df['preceding_job_number'] = df['preceding_job_number'].fillna(0)

### Finalizing Dataset

In [13]:
features = [
    'requested_processors', 
    'requested_time', 
    'queue_number', 
    'think_time', 
    'preceding_job_number', 
    'current_cpu_utilization'
]

target = 'wait_time'

In [14]:
df_final = df[features + [target]]

In [15]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 6452617 entries, 0 to 6452616
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   requested_processors     int64  
 1   requested_time           int64  
 2   queue_number             int64  
 3   think_time               float64
 4   preceding_job_number     int64  
 5   current_cpu_utilization  float64
 6   wait_time                int64  
dtypes: float64(2), int64(5)
memory usage: 344.6 MB


In [16]:
df_final.describe()

,requested_processors,requested_time,queue_number,think_time,preceding_job_number,current_cpu_utilization,wait_time
count,6.452617e+06,6.452617e+06,6.452617e+06,6.452617e+06,6.452617e+06,6.452617e+06,6.452617e+06
mean,4.295811e+00,8.155702e+04,2.745776e+00,3.369688e+03,5.620109e+02,6.928791e+02,1.253623e+04
std,1.504043e+01,1.217488e+06,1.312119e+00,2.723587e+05,7.902795e+02,3.138002e+02,7.169695e+04
min,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,7.140000e+03,2.000000e+00,0.000000e+00,5.800000e+01,4.700000e+02,1.100000e+01
50%,1.000000e+00,9.000000e+03,2.000000e+00,0.000000e+00,2.540000e+02,6.800000e+02,4.600000e+01
75%,1.000000e+00,1.620000e+04,3.000000e+00,0.000000e+00,7.860000e+02,8.980000e+02,2.787000e+03
max,5.120000e+02,3.600000e+07,6.000000e+00,2.237191e+08,9.913000e+03,1.810000e+03,1.057163e+07


In [ ]:
df_final.to_parquet('../dataset/df_final.parquet', index=False)